# Where transfer fails

> Why the zero-shot K predictions are too spread out, and why the pretrained features fail on some instruments

In [ ]:
import numpy as np, pandas as pd
from fastcore.test import *
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from soilspecdata.datasets.ossl import load_ossl
from soilspecdata.datasets.ringtrial import load_ringtrial
from soilspecdl.all import predict, get_splitter
from ossl_rt_transfer.data import rt_path, load_instruments, ossl_props, ossl_overlap, ossl_matches
from ossl_rt_transfer.pretrain import ossl_xy, load_pretrained, rt_wns, model_dir
from ossl_rt_transfer.finetune import rt_xy, cached, repeat_scores
from ossl_rt_transfer.instrument_model import fit_instrument, residuals, paired_spectra

/Users/franckalbinet/aai-ws/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


The cells that read the ring-trial data are marked `#| rtdata`. The outputs on this page come from a local run.

`09_all_labs` left two questions open:

- The zero-shot K predictions are too spread out on 19 of the 21 instruments, KSSL included. Does this come from the model, from the ring-trial scans, or from the soils?
- On UIUC, Lesotho, IAEA and USP, the pretrained clay features fail, and fine-tuning gains most. What makes these instruments different?

All the models here are the unaugmented 80-epoch models of `01_pretrain`.

## The spread of the K predictions

The K model is scored on its own OSSL validation set: the 10% of the OSSL samples that `pretrain` held out, with the same seed. `cal` regresses the targets on the predictions. A calibrated model has a slope of 1. K is on the `log1p` scale throughout:

In [ ]:
#| rtdata
om = load_ossl().mir(650, 4000)
X, y, ids = ossl_xy(om, 'potassium_cmolkg')
tr, va, _ = get_splitter(0.9, 0.1, seed=42)(X, y)
mK = load_pretrained('ossl_potassium_cmolkg_snv_ep80')
pv, yv = predict(mK, X[va], rt_wns).ravel(), y[va]
rt = load_ringtrial(rt_path())
Xk, yk, idk = rt_xy(rt['KSSL'], 'potassium_cmolkg')
pk = pd.Series(predict(mK, Xk, rt_wns).ravel(), index=idk)

def cal(p, t):
    "Slope of `t` on `p`, r², spreads and bias"
    return dict(n=len(t), slope=np.polyfit(p, t, 1)[0], r2=np.corrcoef(p, t)[0, 1]**2, sd_y=t.std(), sd_pred=p.std(), bias=(p - t).mean())

The validation set as a whole, the validation soils with K in the range of the ring-trial soils, the validation soils spectrally nearest to the ring-trial soils, and the ring-trial soils on the KSSL instrument. The nearest soils are the 20 nearest validation soils to each ring-trial soil, in a PCA with 20 components fitted on the SNV spectra of the training set:

In [ ]:
#| rtdata
snv = lambda A: (A - A.mean(1, keepdims=True)) / A.std(1, keepdims=True)
pca = PCA(20, random_state=0).fit(snv(X[tr]))
Zk = pca.transform(snv(Xk))
_, ix = NearestNeighbors(n_neighbors=20).fit(pca.transform(snv(X[va]))).kneighbors(Zk)
near = np.unique(ix.ravel())
in_rng = (yv >= yk.min()) & (yv <= yk.max())
pd.DataFrame({'OSSL validation': cal(pv, yv), 'K in ring-trial range': cal(pv[in_rng], yv[in_rng]),
              'nearest to ring-trial soils': cal(pv[near], yv[near]), 'ring trial, KSSL': cal(pk.values, yk)}).T.round(3)

,n,slope,r2,sd_y,sd_pred,bias
OSSL validation,5763.0,1.020,0.822,0.380,0.338,0.003
K in ring-trial range,5048.0,0.927,0.769,0.289,0.273,0.006
nearest to ring-trial soils,702.0,1.001,0.814,0.342,0.308,0.009
"ring trial, KSSL",69.0,0.561,0.630,0.239,0.338,0.047


37 ring-trial soils are in OSSL with the same K, and `01_pretrain` left them out of training. Their OSSL spectra give a second prediction of the same soils, from the scans of the OSSL library instead of the ring trial:

In [ ]:
#| rtdata
mt = ossl_matches(rt['KSSL'], om)
ov = mt[mt.same_props & mt.ossl_id.isin(ossl_overlap)]
both = [s for s in ov.index if s in pk.index]
rest = [s for s in pk.index if s not in both]
po = predict(mK, om.spectra[pd.Index(om.sample_ids).get_indexer(ov.loc[both, 'ossl_id'])], rt_wns).ravel()
kser = pd.Series(yk, index=idk)
test_eq(len(both), 37)
pd.DataFrame({'OSSL spectra, 37 soils': cal(po, kser[both].values), 'ring-trial spectra, same soils': cal(pk[both].values, kser[both].values),
              'ring-trial spectra, other 32 soils': cal(pk[rest].values, kser[rest].values)}).T.round(3)

,n,slope,r2,sd_y,sd_pred,bias
"OSSL spectra, 37 soils",37.0,0.535,0.816,0.195,0.330,0.114
"ring-trial spectra, same soils",37.0,0.515,0.818,0.195,0.343,0.126
"ring-trial spectra, other 32 soils",32.0,0.719,0.621,0.281,0.308,-0.045


In [ ]:
#| rtdata
np.corrcoef(po, pk[both].values)[0, 1].round(4)

np.float64(0.9987)

The K of each soil's 50 nearest OSSL training soils gives what the library says about soils with these spectra, without the CNN. A mean over neighbours shrinks towards the overall mean, so the same figures on 3,000 validation soils give the benchmark:

In [ ]:
#| rtdata
nn = NearestNeighbors(n_neighbors=50).fit(pca.transform(snv(X[tr])))
nb_rt = y[tr][nn.kneighbors(Zk)[1]].mean(1)
sub = np.random.RandomState(0).choice(len(va), 3000, replace=False)
nb_va = y[tr][nn.kneighbors(pca.transform(snv(X[va][sub])))[1]].mean(1)
pd.DataFrame({'ring trial: K on neighbour K': cal(nb_rt, yk), 'validation: K on neighbour K': cal(nb_va, yv[sub]),
              'ring trial: neighbour K on prediction': cal(pk.values, nb_rt), 'validation: neighbour K on prediction': cal(pv[sub], nb_va)}).T.round(3)

,n,slope,r2,sd_y,sd_pred,bias
ring trial: K on neighbour K,69.0,0.760,0.451,0.239,0.211,0.005
validation: K on neighbour K,3000.0,1.100,0.553,0.377,0.255,0.011
ring trial: neighbour K on prediction,69.0,0.475,0.578,0.211,0.338,0.041
validation: neighbour K on prediction,3000.0,0.634,0.683,0.255,0.332,-0.007


The correlations of the error (prediction minus target) with pH, on the ring-trial soils and on the validation soils that have a pH:

In [ ]:
#| rtdata
ph_rt = rt['KSSL'].source.loc[idk, 'pH_H20'].astype(float).values
_, ph, phid = om.xy(ossl_props['pH_H20'])
ph_va = pd.Series(ph.ravel(), index=phid).reindex(ids[va]).values
ok = ~np.isnan(ph_va)
def pH_corr(p, t, h): return dict(n=len(t), K_pH=np.corrcoef(t, h)[0, 1], pred_pH=np.corrcoef(p, h)[0, 1], error_pH=np.corrcoef(p - t, h)[0, 1])
pd.DataFrame({'ring trial, KSSL': pH_corr(pk.values, yk, ph_rt), 'OSSL validation': pH_corr(pv[ok], yv[ok], ph_va[ok])}).T.round(2)

,n,K_pH,pred_pH,error_pH
"ring trial, KSSL",69.0,0.46,0.63,0.48
OSSL validation,5351.0,0.21,0.21,-0.05


### Results

**The over-dispersion comes from the 70 soils, not from the model or the ring-trial scans.** On its validation set, the K model is calibrated: the slope is 1.02 overall, 0.93 with K in the ring-trial range, and 1.00 on the 702 validation soils nearest to the ring-trial soils. On the ring-trial soils it is 0.56.

**The scans do not matter.** On the 37 ring-trial soils that are also in OSSL, the predictions from their OSSL spectra and from their ring-trial spectra correlate at 0.999. The slopes are 0.54 and 0.52.

**The library agrees with the model.** The K of each soil's 50 nearest OSSL training soils is also too spread out for these soils. The slope of the targets on the neighbours' K is 0.76, against 1.10 on validation soils. The K of these soils varies less than the K of OSSL soils with the same spectra.

**The error grows with pH on these soils only.** On the ring-trial soils, K correlates with pH at 0.46, and the error at 0.48. On the validation soils, K correlates with pH at 0.21, and the error does not depend on pH.

The 70 soils are 60 KSSL quality-control samples and 10 NAPT proficiency samples, not a random sample of OSSL. The 37 soils also in OSSL are more over-dispersed (slope 0.52) than the other 32 (0.72). No change to pretraining can remove a departure that is specific to a set of soils. A local slope and bias, which `08_recalibrate` and `09_all_labs` fit, is what removes it.

## The instruments where the pretrained features fail

The decomposition of `03_instrument_model`, on all 20 instruments. For each instrument, a model with shared parameters turns the KSSL spectrum of each soil into that instrument's spectrum, with all five effects: gain, baseline, shift, broadening and contrast. `shared` is the RMS of the mean SNV residual over soils, the part that a model with shared parameters can remove. `soil` is the RMS of what remains after the mean is removed. `none` is without any effect, `full` with all five. The fits take about 2 minutes and are cached:

In [ ]:
#| rtdata
wn = rt['KSSL'].wavenumbers.astype(float)
full = ('gain', 'baseline', 'shift', 'broaden', 'contrast')
def decompose():
    rows = {}
    for k in [k for k in rt if k != 'KSSL']:
        _, Xi, Xr = paired_spectra(rt, k)
        for name, eff in [('none', ()), ('full', full)]:
            r = residuals(fit_instrument(wn, Xr, Xi, eff), wn, Xr, Xi)
            rows[(k, name)] = dict(shared=np.sqrt((r.mean(0)**2).mean()), soil=np.sqrt(((r - r.mean(0))**2).mean()))
    d = pd.DataFrame(rows).T.unstack()
    d.columns = [f'{b}_{a}' for a, b in d.columns]
    return d
dec = cached('decompose', decompose, model_dir()/'diagnostics')

With the scores of `09_all_labs`, from its saved predictions: the squared zero-shot correlation `r²_zs`, the R² of the head refit, and the gain of fine-tuning over the head refit:

In [ ]:
#| rtdata
d9, props = model_dir()/'finetune', ['potassium_cmolkg', 'clay_perc']
P = pd.concat([pd.read_csv(d9/f'{f}.csv', index_col=0).assign(instrument=i, property=p)
               for p in props for i in rt for f in (f'recal_{i}_{p}_snv_ep80', f'cnn_{i}_{p}_snv_ep80', f'plsr_{i}_{p}')], ignore_index=True)
keys = ['instrument', 'property', 'method']
R2 = repeat_scores(P, keys).groupby(keys).mean()['R²']
zs = P[(P.method == 'zero-shot') & (P.repeat == 0)].groupby(['property', 'instrument']).apply(lambda d: np.corrcoef(d.pred, d.y)[0, 1]**2, include_groups=False)
J = dec[['full_shared', 'full_soil']].copy()
for p, tag in [('clay_perc', 'clay'), ('potassium_cmolkg', 'K')]:
    r = R2.xs(p, level='property').unstack('method')
    J[f'{tag}_r²_zs'], J[f'{tag}_head'], J[f'{tag}_gain'] = zs[p], r['head refit'], r['fine-tuned'] - r['head refit']
J.sort_values('full_soil', ascending=False).round(3)

,full_shared,full_soil,clay_r²_zs,clay_head,clay_gain,K_r²_zs,K_head,K_gain
Lesotho,0.209,0.362,0.101,0.347,0.192,0.002,0.236,0.075
IAEA_aug2022,0.105,0.219,0.281,0.579,0.108,0.471,0.463,0.045
UWisc_fine,0.039,0.186,0.605,0.782,0.023,0.364,0.340,0.197
USP,0.073,0.177,0.319,0.610,0.077,0.614,0.453,0.117
UIUC,0.067,0.170,0.262,0.225,0.475,0.128,-0.142,0.622
Hokudai,0.100,0.161,0.613,0.827,0.001,0.559,0.656,0.083
Woodwell_vertex,0.082,0.154,0.728,0.891,-0.003,0.526,0.599,0.026
OSU,0.062,0.138,0.520,0.763,0.013,0.436,0.576,0.054
Agrocares,0.067,0.138,0.696,0.810,0.033,0.526,0.597,0.057
ETH_alpha_2,0.072,0.135,0.640,0.747,0.034,0.518,0.565,0.099


Spearman correlations over the 20 instruments, with and without Lesotho:

In [ ]:
#| rtdata
def spear(J): return pd.DataFrame({c: {v: spearmanr(J[c], J[v])[0] for v in ['full_shared', 'full_soil']} for c in J.columns[2:]})
pd.concat({'all': spear(J), 'without Lesotho': spear(J.drop('Lesotho'))}).round(2)

clay_r²_zs  clay_head  ...  K_head  K_gain
all             full_shared       -0.74      -0.62  ...   -0.17   -0.06
                full_soil         -0.84      -0.72  ...   -0.41    0.14
without Lesotho full_shared       -0.70      -0.57  ...   -0.05   -0.02
                full_soil         -0.81      -0.68  ...   -0.31    0.21

[4 rows x 6 columns]

Instruments with the same hardware:

In [ ]:
#| rtdata
hw = load_instruments()[['manufacturer', 'model', 'accessory']]
pairs = ['USP', 'Agrocares', 'ETH_alpha_1', 'Woodwell_alpha', 'UIUC', 'OSU']
hw.loc[pairs].join(J[['full_soil', 'clay_r²_zs', 'clay_head']]).round(2)

,manufacturer,model,accessory,full_soil,clay_r²_zs,clay_head
organization,,,,,,
USP,Bruker,Alpha I,Bruker Alpha QuickSnap,0.18,0.32,0.61
Agrocares,Bruker,Alpha I,Bruker Alpha QuickSnap,0.14,0.70,0.81
ETH_alpha_1,Bruker,Alpha I,Bruker Alpha QuickSnap,0.13,0.51,0.68
Woodwell_alpha,Bruker,Alpha II,Bruker Alpha QuickSnap,0.13,0.80,0.85
UIUC,Thermo Fisher,Nicolet iS50,"Pike X, Y Autosampler",0.17,0.26,0.22
OSU,Thermo Fisher,Nicolet iS50,"Pike X,Y Autosampler",0.14,0.52,0.76


### Results

**The four instruments where fine-tuning gains most for clay are among those with the largest soil-specific difference from KSSL.** After the shared effects, the soil-specific RMS is 0.36 on Lesotho, 0.22 on IAEA, 0.19 on UWisc, 0.18 on USP and 0.17 on UIUC, against 0.09 to 0.16 on the other 15.

**For clay, the soil-specific difference tracks the failure of the pretrained features.** Its Spearman correlation over the 20 instruments is -0.84 with the zero-shot r², -0.72 with the R² of the head refit, and +0.73 with the gain of fine-tuning. Without Lesotho, it is -0.81, -0.68 and +0.69. The shared difference correlates less, and the two are not independent. For K, the correlations are weak.

**Hardware does not explain it.** USP has the hardware of Agrocares, ETH_alpha_1 and Woodwell_alpha, a Bruker Alpha with the QuickSnap accessory, but a larger soil-specific difference (0.18 against 0.13 to 0.14) and a lower clay head refit (0.61 against 0.68 to 0.85). UIUC and OSU are both a Thermo Fisher Nicolet iS50 with a Pike X,Y autosampler, with differences of 0.17 and 0.14 and clay head refits of 0.22 and 0.76. UWisc has a large difference but a good head refit for clay (0.78).

**The difference comes from how each lab measured the soils.** Woodwell milled and split all the soils centrally (Safanelli et al., 2023), so particle size is the same in every lab. What remains is on the lab side: the packing and loading of the cup, moisture at scan time, settling of coarse and fine particles in the vial, and the set-up of the optics. `03_instrument_model` linked the soil-specific part to a quartz component, larger on sandy soils, and a water component, larger on clay soils. Which of these causes dominates is not tested here.

## The IAEA shift and the Fukushima scans

`ossl-fuku-transfer/16_moisture` found a broad shift in the Fukushima spectra, which IAEA measured on the same Thermo Fisher Nicolet iS20. After SNV, the Fukushima spectra are higher between 2800 and 3700 cm⁻¹, and lower below 2000 cm⁻¹, than their OSSL soil analogues and than the Bruker spectra of the same soils. On the ring-trial soils, the shift of each instrument from KSSL can be measured on soils that Woodwell prepared centrally. The table gives IAEA's mean SNV difference from KSSL in the regions of `16_moisture`, and the share of soils where IAEA is higher:

In [ ]:
#| rtdata
regions = {'650–1300': (650, 1300), '1300–1610': (1300, 1610), 'H–O–H bend 1610–1650': (1610, 1650), '1650–2000': (1650, 2000), '2000–2800': (2000, 2800),
           '2800–3200': (2800, 3200), 'O–H stretch 3200–3450': (3200, 3450), '3450–3700': (3450, 3700), '3700–4000': (3700, 4000)}
def region_diff(k):
    "Mean SNV difference of `k` from KSSL in each region, and the share of soils where `k` is higher"
    _, Xi, Xr = paired_spectra(rt, k)
    d = snv(Xi) - snv(Xr)
    ms = {reg: (wn >= lo) & (wn <= hi) for reg, (lo, hi) in regions.items()}
    return pd.DataFrame({'mean difference': {r: d[:, m].mean() for r, m in ms.items()}, 'share higher': {r: (d[:, m] > 0).mean() for r, m in ms.items()}})
region_diff('IAEA_aug2022').round(3)

,mean difference,share higher
650–1300,-0.534,0.104
1300–1610,-0.372,0.105
H–O–H bend 1610–1650,-0.388,0.018
1650–2000,-0.064,0.413
2000–2800,0.031,0.471
2800–3200,0.441,0.996
O–H stretch 3200–3450,0.566,1.000
3450–3700,0.632,0.994
3700–4000,0.001,0.358


Each instrument's profile over the eight regions up to 3700 cm⁻¹, against the Fukushima Thermo − Bruker profile, copied from the table of `16_moisture`. `r` is their correlation. `amplitude` is the mean difference over 2800–3700 cm⁻¹ minus that over 650–1300 cm⁻¹:

In [ ]:
#| rtdata
fuku_tb = pd.Series({'650–1300': -1.837, '1300–1610': -0.942, 'H–O–H bend 1610–1650': -0.578, '1650–2000': -0.194, '2000–2800': 0.374,
                     '2800–3200': 1.095, 'O–H stretch 3200–3450': 1.291, '3450–3700': 1.403}) # Thermo − Bruker, `ossl-fuku-transfer/16_moisture`
MD = pd.DataFrame({k: region_diff(k)['mean difference'] for k in rt if k != 'KSSL'})
hi_regs = ['2800–3200', 'O–H stretch 3200–3450', '3450–3700']
prof = pd.DataFrame({'r': MD.loc[fuku_tb.index].apply(lambda c: np.corrcoef(c, fuku_tb)[0, 1]),
                     'amplitude': MD.loc[hi_regs].mean() - MD.loc['650–1300']}).join(hw[['manufacturer', 'accessory']])
prof.sort_values('amplitude', ascending=False).round(2)

,r,amplitude,manufacturer,accessory
Lesotho,0.94,2.68,PerkinElmer,DRIFT
IAEA_aug2022,0.97,1.08,Thermo Fisher,Collector II Diffuse Reflectance Accessory
UWisc_fine,0.68,0.51,Bruker,Bruker Front-facing accessory
CSU_IL,0.15,-0.03,Bruker,Bruker HTS-XT
UIUC,-0.44,-0.06,Thermo Fisher,"Pike X, Y Autosampler"
MSU,-0.57,-0.07,Bruker,Bruker HTS-XT
LandCare,-0.79,-0.13,Bruker,Bruker HTS-XT
Woodwell_alpha,-0.04,-0.13,Bruker,Bruker Alpha QuickSnap
USP,0.20,-0.13,Bruker,Bruker Alpha QuickSnap
Rothamsted,-0.22,-0.15,Bruker,HTS-XT


### Results

**The Thermo shift of the Fukushima scans is also in IAEA's ring-trial scans.** On the ring-trial soils, IAEA is higher than KSSL between 2800 and 3700 cm⁻¹ on 99 to 100% of the soils, by 0.44 to 0.63. It is lower below 1610 cm⁻¹ on 90% of the soils, by 0.37 to 0.53. As in the Fukushima scans, the H–O–H bending band is lower, here on 98% of the soils. Over the eight regions, the IAEA profile correlates with the Fukushima Thermo − Bruker profile at 0.97. Its amplitude is 1.08. In `16_moisture`, the amplitude of the Fukushima spectra against their OSSL analogues is 1.19.

**The shift comes from the IAEA measurement, not from the Fukushima samples.** Woodwell prepared the ring-trial soils, and IAEA scanned them with the instrument that scanned the Fukushima soils. The shift belongs to what the two sets of scans share: the instrument, its Collector II accessory, or how IAEA presented the samples.

**Other Thermo Fisher Nicolets do not show it.** UIUC and OSU, both a Nicolet iS50 with a Pike X,Y autosampler, have profiles that do not match (r -0.44 and -0.46) and amplitudes near zero. The shift is not a property of Nicolet instruments. Lesotho, a PerkinElmer with a DRIFT accessory, has the same profile (r 0.94) with a larger amplitude (2.68).

**The shift is shared by all soils.** A correction with shared parameters can remove a shift that appears on almost every soil. In `03_instrument_model`, gain and baseline reduce the shared difference of IAEA from 0.42 to 0.11. `16_moisture` found that the water-band scores do not explain the Fukushima Kex errors. On the ring-trial soils, the soil-specific difference, not this shift, tracks the failure of the pretrained features.